# Sri Lanka Employment Ratio Analysis (1994–2025)

**Author:** Kalana  
**Source dataset:** `Dataset_Management/SriLanka_Migration_final.csv` (repo root)  
**Output directory:** `analysis/Kalana/final_column_analysis/employment_ratio/`  
**Charts directory:** `analysis/Kalana/charts/employment_ratio/`

## Purpose
Build reproducible annual and monthly employment-ratio datasets for Sri Lanka (1994–2025) directly from the master migration CSV, validate them, produce analytical summaries, and export publication-ready charts.

## Outputs

| File | Description |
|------|-------------|
| `employment_ratio_annual_1994_2025.csv` | 32 annual observations (1994–2025) |
| `employment_ratio_monthly_1994_2025.csv` | 384 monthly rows (linear interpolation) |
| `employment_ratio_summary_stats.csv` | Descriptive statistics |
| `employment_ratio_paper_ready_table_1994_2025.csv` | Rounded table for reports/appendix |
| `charts/employment_ratio/*.png` | PNG charts (timeseries, bars, YoY) |

## Reproducibility
Run from **any** directory inside the repo clone. `find_repo_root()` locates the root by searching for `Dataset_Management/SriLanka_Migration_final.csv` — no hardcoded paths.

---\n## 1. Setup and Path Resolution

In [ ]:
import importlib, subprocess, sys

def _ensure(pkg, import_name=None):
    if importlib.util.find_spec(import_name or pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

_ensure("pandas")
_ensure("matplotlib")
_ensure("seaborn")

import csv
import os
from datetime import date
from pathlib import Path
from statistics import mean, pstdev

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# ── Repo-root detection ──────────────────────────────────────────────────────
def find_repo_root(start: Path) -> Path:
    marker = Path("Dataset_Management") / "SriLanka_Migration_final.csv"
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(
        "Cannot locate repo root. Ensure SriLanka_Migration_final.csv exists "
        "under Dataset_Management/ in the repository root."
    )

REPO_ROOT    = find_repo_root(Path.cwd())
MASTER_CSV   = REPO_ROOT / "Dataset_Management" / "SriLanka_Migration_final.csv"
ANALYSIS_DIR = REPO_ROOT / "analysis" / "Kalana" / "final_column_analysis" / "employment_ratio"
CHARTS_DIR   = REPO_ROOT / "analysis" / "Kalana" / "charts" / "employment_ratio"

ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Repo root   : {REPO_ROOT}")
print(f"Master CSV  : {MASTER_CSV}")
print(f"Analysis dir: {ANALYSIS_DIR}")
print(f"Charts dir  : {CHARTS_DIR}")

---\n## 2. Load and Filter Annual Data (1994–2025)\n\nData is loaded directly from the master CSV and aggregated to annual level.

In [ ]:
df_raw = pd.read_csv(MASTER_CSV)
df_raw["date"] = pd.to_datetime(df_raw["date"], errors="coerce")
df_raw["employment_ratio_annual"] = pd.to_numeric(df_raw["employment_ratio_annual"], errors="coerce")
df_raw["year"] = df_raw["date"].dt.year

# Annual values repeat for each month in the master CSV — take mean to get one value per year
annual_df = (
    df_raw[df_raw["year"].between(1994, 2025)]
    .groupby("year", as_index=False)["employment_ratio_annual"]
    .mean()
    .sort_values("year")
    .reset_index(drop=True)
)

annual_values = dict(zip(annual_df["year"], annual_df["employment_ratio_annual"]))
years = sorted(annual_values)

print(f"Years: {years[0]}–{years[-1]}  |  Rows: {len(annual_df)}")
annual_df

---\n## 3. Export Annual Dataset

In [ ]:
annual_path = ANALYSIS_DIR / "employment_ratio_annual_1994_2025.csv"
annual_df.to_csv(annual_path, index=False)
print(f"Saved: {annual_path.relative_to(REPO_ROOT)}  ({len(annual_df)} rows)")

---\n## 4. Build Monthly Dataset via Linear Interpolation\n\nEach year's annual value is treated as the January anchor. Between consecutive January anchors the series is linearly interpolated to produce month-level values.\n\n- `employment_ratio_annual` — annual value repeated for all 12 months of that year  \n- `employment_ratio_monthly` — linearly interpolated intra-year monthly series

In [ ]:
monthly_rows = []
for i, yr in enumerate(years):
    v_curr = annual_values[yr]
    v_next = annual_values[years[i + 1]] if i + 1 < len(years) else v_curr
    for m in range(1, 13):
        frac = (m - 1) / 12.0
        monthly_rows.append({
            "date":                     date(yr, m, 1).isoformat(),
            "year":                     yr,
            "month":                    m,
            "employment_ratio_monthly": round(v_curr + (v_next - v_curr) * frac, 6),
            "employment_ratio_annual":  round(v_curr, 6),
        })

monthly_df = pd.DataFrame(monthly_rows)
monthly_df["date"] = pd.to_datetime(monthly_df["date"])

print(f"Monthly rows: {len(monthly_df)}  |  {monthly_df['date'].min().date()} → {monthly_df['date'].max().date()}")
monthly_df.head()

---\n## 5. Export Monthly Dataset

In [ ]:
monthly_path = ANALYSIS_DIR / "employment_ratio_monthly_1994_2025.csv"
monthly_df.to_csv(monthly_path, index=False)
print(f"Saved: {monthly_path.relative_to(REPO_ROOT)}  ({len(monthly_df)} rows)")

---\n## 6. Data Validation

In [ ]:
assert len(monthly_df) == 384, f"Expected 384 rows, got {len(monthly_df)}"
assert str(monthly_df["date"].min().date()) == "1994-01-01", "Start date mismatch"
assert str(monthly_df["date"].max().date()) == "2025-12-01", "End date mismatch"
assert monthly_df.isnull().sum().sum() == 0, "Unexpected missing values"

print("All validation checks passed")
print(f"  Rows          : {len(monthly_df)}")
print(f"  Date range    : {monthly_df['date'].min().date()} to {monthly_df['date'].max().date()}")
print(f"  Missing values: {monthly_df.isnull().sum().sum()}")

---\n## 7. Descriptive Statistics and Trend Analysis

In [ ]:
vals = monthly_df["employment_ratio_monthly"].tolist()

summary = {
    "mean":           round(mean(vals), 4),
    "std_dev":        round(pstdev(vals), 4),
    "min":            round(min(vals), 4),
    "max":            round(max(vals), 4),
    "overall_change": round(vals[-1] - vals[0], 4),
    "n_months":       len(vals),
}

print("Summary Statistics (monthly interpolated series)")
print("-" * 45)
for k, v in summary.items():
    print(f"  {k:<20}: {v}")

# Year-over-year change from annual series
annual_df["yoy_change"] = annual_df["employment_ratio_annual"].diff()
print("\nTop 5 annual increases:")
print(annual_df.nlargest(5, "yoy_change")[["year", "employment_ratio_annual", "yoy_change"]].to_string(index=False))
print("\nTop 5 annual decreases:")
print(annual_df.nsmallest(5, "yoy_change")[["year", "employment_ratio_annual", "yoy_change"]].to_string(index=False))

---\n## 8. Additional Analytical Views\n\nYearly averages computed from the monthly interpolated series, and top annual swings.

In [ ]:
yearly_avg = monthly_df.groupby("year")["employment_ratio_monthly"].mean().reset_index()
yearly_avg.columns = ["year", "avg_monthly_interpolated"]

anchor_changes = []
for i in range(1, len(annual_df)):
    prev = annual_df.iloc[i - 1]
    curr = annual_df.iloc[i]
    anchor_changes.append({
        "from_year": int(prev["year"]),
        "to_year":   int(curr["year"]),
        "change":    round(curr["employment_ratio_annual"] - prev["employment_ratio_annual"], 4),
    })

changes_df = pd.DataFrame(anchor_changes)
print("Top 5 increases:")
print(changes_df.nlargest(5, "change").to_string(index=False))
print("\nTop 5 decreases:")
print(changes_df.nsmallest(5, "change").to_string(index=False))

---\n## 9. Export Summary Statistics

In [ ]:
stats_path = ANALYSIS_DIR / "employment_ratio_summary_stats.csv"
pd.DataFrame([summary]).to_csv(stats_path, index=False)
print(f"Saved: {stats_path.relative_to(REPO_ROOT)}")

---\n## 10. Visualisations\n\nAll charts are saved to `analysis/Kalana/charts/employment_ratio/`.

In [ ]:
sns.set_theme(style="whitegrid")

# ── Chart 1: Monthly Time Series ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(monthly_df["date"], monthly_df["employment_ratio_monthly"],
        color="#1F618D", linewidth=1.8, label="Monthly (interpolated)")
ax.plot(annual_df["year"], annual_df["employment_ratio_annual"],
        "o", color="#C0392B", markersize=5, label="Annual anchor")
ax.set_title("Sri Lanka Employment Ratio — Monthly Trend (1994–2025)", fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Employment Ratio (%)")
ax.legend()
fig.tight_layout()
fig.savefig(CHARTS_DIR / "employment_ratio_monthly_timeseries.png", dpi=180)
plt.show()
print(f"Saved: charts/employment_ratio/employment_ratio_monthly_timeseries.png")

# ── Chart 2: Yearly Average Bars ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(yearly_avg["year"], yearly_avg["avg_monthly_interpolated"],
       color="#1F618D", edgecolor="white", linewidth=0.5)
ax.set_title("Sri Lanka Employment Ratio — Yearly Average (1994–2025)", fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Employment Ratio (%)")
ax.set_xticks(yearly_avg["year"])
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "employment_ratio_yearly_average_bars.png", dpi=180)
plt.show()
print(f"Saved: charts/employment_ratio/employment_ratio_yearly_average_bars.png")

# ── Chart 3: Year-over-Year Change ───────────────────────────────────────────
yoy = annual_df.dropna(subset=["yoy_change"])
colors = ["#27AE60" if v >= 0 else "#C0392B" for v in yoy["yoy_change"]]
fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(yoy["year"], yoy["yoy_change"], color=colors, edgecolor="white", linewidth=0.4)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("Year-over-Year Change in Employment Ratio (1994–2025)", fontsize=13, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Change (pp)")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(CHARTS_DIR / "employment_ratio_yoy_change.png", dpi=180)
plt.show()
print(f"Saved: charts/employment_ratio/employment_ratio_yoy_change.png")

---\n## 11. Paper-Ready Export Table

In [ ]:
month_names = {1:"Jan",2:"Feb",3:"Mar",4:"Apr",5:"May",6:"Jun",
               7:"Jul",8:"Aug",9:"Sep",10:"Oct",11:"Nov",12:"Dec"}

paper_df = monthly_df.copy()
paper_df["period"]     = paper_df["date"].dt.to_period("M").astype(str)
paper_df["month_name"] = paper_df["month"].map(month_names)
paper_df["quarter"]    = paper_df["month"].apply(lambda m: f"Q{(m - 1) // 3 + 1}")
paper_df["employment_ratio_monthly"] = paper_df["employment_ratio_monthly"].round(3)
paper_df["employment_ratio_annual"]  = paper_df["employment_ratio_annual"].round(3)

cols = ["period","year","month","month_name","quarter","employment_ratio_monthly","employment_ratio_annual"]
paper_out = ANALYSIS_DIR / "employment_ratio_paper_ready_table_1994_2025.csv"
paper_df[cols].to_csv(paper_out, index=False)
print(f"Saved: {paper_out.relative_to(REPO_ROOT)}  ({len(paper_df)} rows)")
paper_df[cols].head(12)

---\n## 12. Final Notes\n\n| Item | Detail |\n|------|--------|\n| Source | `Dataset_Management/SriLanka_Migration_final.csv` |\n| Annual rows | 32 (1994–2025) |\n| Monthly rows | 384 (Jan 1994 – Dec 2025) |\n| Interpolation | Linear between consecutive January anchors |\n| Charts | `analysis/Kalana/charts/employment_ratio/` |\n| `employment_ratio_annual` | Annual value repeated for every month of that year |\n| `employment_ratio_monthly` | Smoothly interpolated month-level series |